### Dataset Input

In [1]:

from pathlib import Path

# Locate the existing project root
BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DATASET_DIR = RAW_DIR / "spacenet8"

# Compatibility aliases for later lab cells
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("SpaceNet 8 directory:", DATASET_DIR)

assert DATASET_DIR.is_dir(), "SpaceNet 8 dataset folder not found."
print("Workspace initialized successfully.")


Project root: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard
Raw data directory: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard\data\raw
Processed data directory: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard\data\processed
SpaceNet 8 directory: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard\data\raw\spacenet8
Workspace initialized successfully.


In [2]:

from collections import Counter

# Find all extracted files
all_files = sorted(
    file for file in DATASET_DIR.rglob("*")
    if file.is_file()
)

file_types = Counter(
    file.suffix.lower() for file in all_files
)

print("Total files:", len(all_files))
print("\nFile types:")
for ext, count in sorted(file_types.items()):
    print(f"{ext or '[no extension]'}: {count}")

print("\nSample files:")
for file in all_files[:15]:
    print(file.relative_to(DATASET_DIR))

# Verify expected folders
for folder in ["PRE-event", "POST-event", "annotations"]:
    path = DATASET_DIR / folder
    print(f"\n{folder}: {'Found' if path.is_dir() else 'Missing'}")

assert all_files, "No input files found."
print("\nInput verification completed.")


Total files: 689

File types:
.csv: 2
.geojson: 202
.gz: 1
.tif: 484

Sample files:
annotations\0_15_63.geojson
annotations\0_15_65.geojson
annotations\0_15_66.geojson
annotations\0_15_67.geojson
annotations\0_15_68.geojson
annotations\0_15_69.geojson
annotations\0_15_70.geojson
annotations\0_16_63.geojson
annotations\0_16_66.geojson
annotations\0_16_67.geojson
annotations\0_16_68.geojson
annotations\0_16_70.geojson
annotations\0_17_63.geojson
annotations\0_17_66.geojson
annotations\0_17_67.geojson

PRE-event: Found

POST-event: Found

annotations: Found

Input verification completed.


In [3]:

pre_images = sorted((DATASET_DIR / "PRE-event").rglob("*.tif"))
post_images = sorted((DATASET_DIR / "POST-event").rglob("*.tif"))
annotations = sorted((DATASET_DIR / "annotations").rglob("*.geojson"))

print("PRE-event images:", len(pre_images))
print("POST-event images:", len(post_images))
print("GeoJSON annotations:", len(annotations))

print("\nExample PRE-event image:")
print(pre_images[0].relative_to(DATASET_DIR) if pre_images else "None")

print("\nExample POST-event image:")
print(post_images[0].relative_to(DATASET_DIR) if post_images else "None")

print("\nExample annotation:")
print(annotations[0].relative_to(DATASET_DIR) if annotations else "None")


PRE-event images: 202
POST-event images: 282
GeoJSON annotations: 202

Example PRE-event image:
PRE-event\10500500C4DD7000_0_15_63.tif

Example POST-event image:
POST-event\1040050035DC3B00_0_15_63.tif

Example annotation:
annotations\0_15_63.geojson


In [4]:

import rasterio

def inspect_geotiff(path):
    with rasterio.open(path) as src:
        return {
            "file": str(path.relative_to(DATASET_DIR)),
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "dtype": src.dtypes,
            "crs": str(src.crs),
            "resolution": src.res,
            "nodata": src.nodata,
        }

if pre_images and post_images:
    pre_info = inspect_geotiff(pre_images[0])
    post_info = inspect_geotiff(post_images[0])

    print("PRE-EVENT IMAGE")
    for key, value in pre_info.items():
        print(f"{key}: {value}")

    print("\nPOST-EVENT IMAGE")
    for key, value in post_info.items():
        print(f"{key}: {value}")


PRE-EVENT IMAGE
file: PRE-event\10500500C4DD7000_0_15_63.tif
width: 1300
height: 1300
bands: 3
dtype: ('uint8', 'uint8', 'uint8')
crs: EPSG:4326
resolution: (3.74298035050102e-06, 3.74298035050102e-06)
nodata: 255.0

POST-EVENT IMAGE
file: POST-event\1040050035DC3B00_0_15_63.tif
width: 1743
height: 1742
bands: 3
dtype: ('uint8', 'uint8', 'uint8')
crs: EPSG:4326
resolution: (2.793760533611708e-06, 2.793760533611708e-06)
nodata: None
